# LCOE Alternative Scenarios

This standalone notebook reproduces the minimum baseline LCOE mechanics from `lcoe.ipynb` and the financing structure from `cost-of-capital.ipynb`, then applies additional sensitivities requested for CAPEX, equity, debt, leverage, and OPEX.

It **does not modify** any baseline notebook or baseline output file.


## Inputs inspected
- `lcoe_solar_analysis.csv` (project-level baseline LCOE inputs/outputs)
- `bndes_wacc_calculations.csv` (monthly financing components: cost of debt, cost of equity, debt share, and technology premium)

## Scenario implementation notes
- Technology premium is separately observable in `bndes_wacc_calculations.csv`, so `tech_premium_half` is implemented.
- Debt share cap uses a hard ceiling of **85%** for `debt_share_plus_10pp` and `combined_financing_relief`.
- OPEX is accessible as a project-level `opex_inflated` list, so `opex_minus_10` is implemented.


In [ ]:
import ast
import csv
import math
import statistics
from collections import defaultdict
from datetime import datetime
from pathlib import Path

LIFETIME_YEARS = 25
INPUT_LCOE = Path('lcoe_solar_analysis.csv')
INPUT_BNDES = Path('bndes_wacc_calculations.csv')
OUT_PROJECT = Path('lcoe_alternative_scenarios_results.csv')
OUT_SUMMARY = Path('lcoe_alternative_scenarios_summary.csv')
OUT_FIG = Path('lcoe_alternative_scenarios_by_year.svg')


In [ ]:
def to_float(v):
    try:
        return float(v)
    except (TypeError, ValueError):
        return math.nan


def parse_opex_list(v):
    try:
        parsed = ast.literal_eval(v)
        if isinstance(parsed, list):
            return [float(x) for x in parsed]
    except (ValueError, SyntaxError, TypeError):
        pass
    return []


def modeled_lcoe(exp_energy_prod, capex, opex_inflated, wacc_pct, lifetime=LIFETIME_YEARS):
    if not (exp_energy_prod > 0 and capex >= 0 and len(opex_inflated) > 0):
        return math.nan

    wacc_decimal = wacc_pct / 100.0
    if wacc_decimal <= -1:
        return math.nan

    discounted_energy = 0.0
    discounted_opex = 0.0

    for t in range(1, lifetime + 1):
        discount = (1 + wacc_decimal) ** t
        discounted_energy += exp_energy_prod / discount
        discounted_opex += opex_inflated[min(t - 1, len(opex_inflated) - 1)] / discount

    if discounted_energy <= 0:
        return math.nan

    return (capex + discounted_opex) / discounted_energy


In [ ]:
# Monthly financing components from cost-of-capital output
finance_lookup = {}
with INPUT_BNDES.open(newline='', encoding='utf-8') as f:
    for r in csv.DictReader(f):
        d = datetime.strptime(r['date'], '%d %B %Y')
        finance_lookup[(d.year, d.month)] = {
            'cost_of_debt': to_float(r.get('cost_of_debt')),
            'cost_of_equity': to_float(r.get('cost_of_equity')),
            'debt_share': to_float(r.get('debt_share')),
            'technology_premium': to_float(r.get('technology_premium')),
            'us_treasury_yield': to_float(r.get('us_treasury_yield')),
            'cds_brazil': to_float(r.get('cds_brazil')),
            'erp_sp500': to_float(r.get('erp_sp500')),
        }

# Project analytical sample from baseline output
projects = []
with INPUT_LCOE.open(newline='', encoding='utf-8') as f:
    for r in csv.DictReader(f):
        year = int(float(r['year'])) if r.get('year') else None
        month = int(float(r['month'])) if r.get('month') else None
        opex = parse_opex_list(r.get('opex_inflated', '[]'))
        if len(opex) == 0:
            continue

        p = {
            'power_plant_name': r.get('power_plant_name', ''),
            'date': r.get('date_x', ''),
            'year': year,
            'month': month,
            'capex': to_float(r.get('capex')),
            'exp_energy_prod': to_float(r.get('exp_energy_prod')),
            'sale_price_auction': to_float(r.get('sale_price_auction')),
            'baseline_wacc': to_float(r.get('cost_of_capital')),
            'opex_inflated': opex,
        }
        p.update(finance_lookup.get((year, month), {}))
        projects.append(p)

print(f'Analytical sample size: {len(projects)} projects')


In [ ]:
SCENARIOS = [
    'baseline',
    'capex_minus_10',
    'capex_minus_15',
    'coe_minus_200bps',
    'tech_premium_half',
    'cod_minus_100bps',
    'cod_minus_200bps',
    'debt_share_plus_10pp',
    'combined_financing_relief',
    'opex_minus_10',
]


def scenario_values(project, scenario):
    capex = project['capex']
    opex = list(project['opex_inflated'])
    wacc = project['baseline_wacc']
    cod = project.get('cost_of_debt', math.nan)
    coe = project.get('cost_of_equity', math.nan)
    debt_share = project.get('debt_share', math.nan)
    tp = project.get('technology_premium', math.nan)
    grf = project.get('us_treasury_yield', math.nan)
    cds = project.get('cds_brazil', math.nan)
    erp = project.get('erp_sp500', math.nan)

    if scenario == 'baseline':
        pass
    elif scenario == 'capex_minus_10':
        capex *= 0.90
    elif scenario == 'capex_minus_15':
        capex *= 0.85
    elif scenario == 'coe_minus_200bps':
        if any(math.isnan(x) for x in [cod, coe, debt_share]):
            return None
        coe = max(coe - 2.0, 0.0)
        wacc = cod * debt_share + coe * (1 - debt_share)
    elif scenario == 'tech_premium_half':
        if any(math.isnan(x) for x in [tp, grf, cds, erp, debt_share]):
            return None
        tp2 = tp * 0.5
        cod = (grf + cds + tp2) * (1 - 0.34)
        coe = grf + cds + erp + tp2
        wacc = cod * debt_share + coe * (1 - debt_share)
    elif scenario == 'cod_minus_100bps':
        if any(math.isnan(x) for x in [cod, coe, debt_share]):
            return None
        cod = max(cod - 1.0, 0.0)
        wacc = cod * debt_share + coe * (1 - debt_share)
    elif scenario == 'cod_minus_200bps':
        if any(math.isnan(x) for x in [cod, coe, debt_share]):
            return None
        cod = max(cod - 2.0, 0.0)
        wacc = cod * debt_share + coe * (1 - debt_share)
    elif scenario == 'debt_share_plus_10pp':
        if any(math.isnan(x) for x in [cod, coe, debt_share]):
            return None
        debt_share = min(debt_share + 0.10, 0.85)
        wacc = cod * debt_share + coe * (1 - debt_share)
    elif scenario == 'combined_financing_relief':
        if any(math.isnan(x) for x in [cod, coe, debt_share]):
            return None
        cod = max(cod - 1.0, 0.0)
        coe = max(coe - 2.0, 0.0)
        debt_share = min(debt_share + 0.10, 0.85)
        wacc = cod * debt_share + coe * (1 - debt_share)
    elif scenario == 'opex_minus_10':
        opex = [x * 0.90 for x in opex]
    else:
        return None

    return capex, opex, wacc, cod, coe, debt_share


In [ ]:
rows = []
for p in projects:
    for scenario in SCENARIOS:
        values = scenario_values(p, scenario)
        if values is None:
            continue

        capex, opex, wacc, cod, coe, debt_share = values
        lcoe = modeled_lcoe(p['exp_energy_prod'], capex, opex, wacc)
        auction = p['sale_price_auction']
        gap = lcoe - auction if not (math.isnan(lcoe) or math.isnan(auction)) else math.nan
        gap_pct = (gap / auction * 100.0) if (auction and not math.isnan(auction) and not math.isnan(gap)) else math.nan

        rows.append({
            'scenario': scenario,
            'power_plant_name': p['power_plant_name'],
            'date': p['date'],
            'year': p['year'],
            'month': p['month'],
            'scenario_wacc_pct': wacc,
            'scenario_lcoe_brl_mwh': lcoe,
            'auction_price_brl_mwh': auction,
            'absolute_gap_brl_mwh': gap,
            'percentage_gap_to_auction_pct': gap_pct,
            'lcoe_leq_auction': int(not math.isnan(lcoe) and not math.isnan(auction) and lcoe <= auction),
            'scenario_capex_brl': capex,
            'scenario_cod_pct': cod,
            'scenario_coe_pct': coe,
            'scenario_debt_share': debt_share,
        })

with OUT_PROJECT.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)

print(f'Saved project-level output: {OUT_PROJECT} (rows={len(rows)})')


In [ ]:
# Compact scenario summary
scenario_groups = defaultdict(list)
for r in rows:
    scenario_groups[r['scenario']].append(r)

summary_rows = []
for scenario in SCENARIOS:
    if scenario not in scenario_groups:
        continue

    group = scenario_groups[scenario]
    lcoe_vals = [x['scenario_lcoe_brl_mwh'] for x in group if not math.isnan(x['scenario_lcoe_brl_mwh'])]
    gap_vals = [x['absolute_gap_brl_mwh'] for x in group if not math.isnan(x['absolute_gap_brl_mwh'])]
    wacc_vals = [x['scenario_wacc_pct'] for x in group if not math.isnan(x['scenario_wacc_pct'])]
    n = len(group)
    n_leq = sum(x['lcoe_leq_auction'] for x in group)

    summary_rows.append({
        'scenario': scenario,
        'n_projects': n,
        'mean_lcoe_brl_mwh': statistics.fmean(lcoe_vals) if lcoe_vals else math.nan,
        'median_lcoe_brl_mwh': statistics.median(lcoe_vals) if lcoe_vals else math.nan,
        'mean_gap_brl_mwh': statistics.fmean(gap_vals) if gap_vals else math.nan,
        'median_gap_brl_mwh': statistics.median(gap_vals) if gap_vals else math.nan,
        'n_lcoe_leq_auction': n_leq,
        'share_lcoe_leq_auction': n_leq / n if n else math.nan,
        'mean_scenario_wacc_pct': statistics.fmean(wacc_vals) if wacc_vals else math.nan,
    })

with OUT_SUMMARY.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=list(summary_rows[0].keys()))
    writer.writeheader()
    writer.writerows(summary_rows)

for s in summary_rows:
    print(f"{s['scenario']:<28} mean_lcoe={s['mean_lcoe_brl_mwh']:.2f}  mean_gap={s['mean_gap_brl_mwh']:.2f}  share<=auction={s['share_lcoe_leq_auction']:.1%}")

print(f'\nSaved summary: {OUT_SUMMARY}')


In [ ]:
# Minimal SVG figure: baseline average LCOE by year vs scenario averages by year
series = defaultdict(lambda: defaultdict(list))
for r in rows:
    series[r['scenario']][r['year']].append(r['scenario_lcoe_brl_mwh'])

avg_series = {
    scenario: {year: statistics.fmean(vals) for year, vals in year_map.items()}
    for scenario, year_map in series.items()
}

years = sorted({y for by_year in avg_series.values() for y in by_year})
scenarios_plot = ['baseline'] + [s for s in SCENARIOS if s != 'baseline' and s in avg_series]

# Simple hand-built SVG (no external plotting libs)
width, height = 980, 520
ml, mr, mt, mb = 70, 240, 30, 60
plot_w, plot_h = width - ml - mr, height - mt - mb
vals = [v for s in scenarios_plot for v in avg_series[s].values()]
ymin, ymax = min(vals), max(vals)
if ymax == ymin:
    ymax = ymin + 1

def xpix(year):
    idx = years.index(year)
    denom = (len(years) - 1) if len(years) > 1 else 1
    return ml + (idx / denom) * plot_w

def ypix(val):
    return mt + (1 - (val - ymin) / (ymax - ymin)) * plot_h

colors = ['#000000','#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd','#8c564b','#e377c2','#17becf','#bcbd22']
parts = [
    f"<svg xmlns='http://www.w3.org/2000/svg' width='{width}' height='{height}'>",
    "<rect width='100%' height='100%' fill='white'/>",
    f"<line x1='{ml}' y1='{mt+plot_h}' x2='{ml+plot_w}' y2='{mt+plot_h}' stroke='black'/>",
    f"<line x1='{ml}' y1='{mt}' x2='{ml}' y2='{mt+plot_h}' stroke='black'/>",
]

for i in range(6):
    val = ymin + (ymax - ymin) * i / 5
    y = ypix(val)
    parts.append(f"<line x1='{ml-5}' y1='{y}' x2='{ml+plot_w}' y2='{y}' stroke='#dddddd'/>")
    parts.append(f"<text x='{ml-10}' y='{y+4}' text-anchor='end' font-size='11'>{val:.1f}</text>")

for y in years:
    x = xpix(y)
    parts.append(f"<line x1='{x}' y1='{mt+plot_h}' x2='{x}' y2='{mt+plot_h+5}' stroke='black'/>")
    parts.append(f"<text x='{x}' y='{mt+plot_h+20}' text-anchor='middle' font-size='11'>{y}</text>")

for idx, scenario in enumerate(scenarios_plot):
    pts = [f"{xpix(y):.1f},{ypix(avg_series[scenario][y]):.1f}" for y in years if y in avg_series[scenario]]
    if len(pts) >= 2:
        parts.append(f"<polyline fill='none' stroke='{colors[idx % len(colors)]}' stroke-width='2' points='{' '.join(pts)}'/>")

legend_x = ml + plot_w + 15
for idx, scenario in enumerate(scenarios_plot):
    yy = mt + 20 + idx * 20
    c = colors[idx % len(colors)]
    parts.append(f"<line x1='{legend_x}' y1='{yy}' x2='{legend_x+20}' y2='{yy}' stroke='{c}' stroke-width='3'/>")
    parts.append(f"<text x='{legend_x+26}' y='{yy+4}' font-size='12'>{scenario}</text>")

parts.append(f"<text x='{width/2}' y='18' text-anchor='middle' font-size='16'>Average LCOE by Year: Baseline vs Alternative Scenarios</text>")
parts.append(f"<text x='{width/2}' y='{height-10}' text-anchor='middle' font-size='12'>Year</text>")
parts.append(f"<text x='18' y='{height/2}' transform='rotate(-90,18,{height/2})' text-anchor='middle' font-size='12'>Average LCOE (BRL/MWh)</text>")
parts.append("</svg>")

OUT_FIG.write_text(''.join(parts), encoding='utf-8')
print(f'Saved figure: {OUT_FIG}')


## Figure output
If this notebook is run in Jupyter, display the SVG below:

```python
from IPython.display import SVG, display
display(SVG(filename='lcoe_alternative_scenarios_by_year.svg'))
```
